# Predição da duração de viagens no sistema Divvy com Apache Spark

**Disciplina:** Processamento de Dados Massivos  
**Problema:** Regressão  
**Ferramentas:** PySpark e Spark ML no Google Colab  
**Fonte:** [Divvy — System Data](https://divvybikes.com/system-data) · [Arquivos mensais](https://divvy-tripdata.s3.amazonaws.com/index.html)  
**Período:** janeiro a dezembro de 2024

## 1. Contextualização e objetivo

O objetivo é prever a **duração de uma viagem, em minutos**, a partir de informações disponíveis no momento do início: tipo de veículo, tipo de usuário, horário e coordenadas de origem. A pergunta de negócio é: **em que medida essas características permitem estimar a duração de uma viagem no sistema Divvy?**

A base pública reúne viagens de bicicletas convencionais, bicicletas elétricas e patinetes elétricos. Os 12 arquivos mensais de 2024 totalizam **5.860.568 registros e 13 colunas**, antes da limpeza. A amostra do trabalho corresponde ao ano completo de 2024, não a uma seleção aleatória de registros.

**Justificativa de Big Data.** A Divvy gera dados continuamente, e análises históricas exigem integração de arquivos, limpeza, agregações e aprendizado de máquina sobre milhões de viagens. O Spark oferece processamento paralelo, particionamento e possibilidade de escalabilidade em clusters. Neste projeto ele é executado **em modo local no Colab**: a base escolhida não exige obrigatoriamente um cluster, mas o pipeline utiliza operações que também podem ser distribuídas.

São comparados **dois algoritmos de regressão** do Spark ML: **Regressão Linear** e **Random Forest Regressor**. Ambos preveem a duração da viagem em minutos e são comparados com um baseline baseado na média do treinamento, usando RMSE, MAE e R². O conjunto de teste é o último trimestre de 2024.

## 2. Configuração e ingestão de dados

### 2.1 Ambiente

A instalação é feita no Colab; a versão do PySpark é fixada para facilitar a reprodução da execução original.

In [1]:
%pip install -q pyspark==4.0.5


In [2]:
from pyspark.sql import SparkSession

# Inicializa uma sessão do Apache Spark
spark = (
    SparkSession.builder
    .appName("Projeto_Regressao_Divvy")
    .master("local[*]")
    .getOrCreate()
)

# Exibe a versão do Spark
print("Versão do Spark:", spark.version)


Versão do Spark: 4.0.5


### 2.2 Download dos arquivos originais

Os CSVs são baixados dos ZIPs oficiais. Os dados **não** precisam ser publicados no GitHub; uma nova execução recria os arquivos em `/content/divvy_2024`.

In [3]:
import os
import requests
import zipfile

diretorio = "/content/divvy_2024"
os.makedirs(diretorio, exist_ok=True)
url_base = "https://divvy-tripdata.s3.amazonaws.com"

for mes in range(1, 13):
    prefixo = f"2024{mes:02d}-divvy-tripdata"
    caminho_csv = os.path.join(diretorio, prefixo + ".csv")
    if os.path.isfile(caminho_csv) and os.path.getsize(caminho_csv) > 0:
        print(f"{prefixo}.csv já disponível")
        continue

    caminho_zip = os.path.join(diretorio, prefixo + ".zip")
    with requests.get(f"{url_base}/{prefixo}.zip", stream=True, timeout=120) as resposta:
        resposta.raise_for_status()
        with open(caminho_zip, "wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    arquivo.write(bloco)

    with zipfile.ZipFile(caminho_zip) as zip_ref:
        zip_ref.extractall(diretorio)
    os.remove(caminho_zip)
    print(f"{prefixo}.csv extraído")

arquivos_csv = sorted(nome for nome in os.listdir(diretorio) if nome.endswith(".csv"))
print("Quantidade de CSVs:", len(arquivos_csv))
assert len(arquivos_csv) == 12, "Esperados 12 arquivos CSV de 2024"


202401-divvy-tripdata.csv extraído
202402-divvy-tripdata.csv extraído
202403-divvy-tripdata.csv extraído
202404-divvy-tripdata.csv extraído
202405-divvy-tripdata.csv extraído
202406-divvy-tripdata.csv extraído
202407-divvy-tripdata.csv extraído
202408-divvy-tripdata.csv extraído
202409-divvy-tripdata.csv extraído
202410-divvy-tripdata.csv extraído
202411-divvy-tripdata.csv extraído
202412-divvy-tripdata.csv extraído
Quantidade de CSVs: 12


### 2.3 Schema e leitura com Spark

O `StructType` define manualmente as **13 colunas**. Na leitura dos CSVs, as datas são recebidas como texto e convertidas com `to_timestamp`, que aceita tanto horários com milissegundos quanto sem eles. Não são adicionadas colunas temporárias.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

schema_divvy = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True)
])

# Schema de ingestão independente: timestamps chegam do CSV como texto.
schema_leitura = StructType([
    StructField(campo.name, StringType(), True)
    if campo.name in ("started_at", "ended_at") else campo
    for campo in schema_divvy.fields
])

df_bruto = (
    spark.read.option("header", True)
    .schema(schema_leitura)
    .csv(f"{diretorio}/*.csv")
)
df = (
    df_bruto
    .withColumn("started_at", F.to_timestamp("started_at"))
    .withColumn("ended_at", F.to_timestamp("ended_at"))
)

df.printSchema()
print("Total de registros:", df.count())
print("Total de colunas:", len(df.columns))
assert len(df.columns) == 13, "O CSV deve conter exatamente 13 colunas"


root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)

Total de registros: 5860568
Total de colunas: 13


In [5]:
# Verificação após a conversão
df.agg(
    F.count("*").alias("total_registros"),
    F.min("started_at").alias("primeira_viagem"),
    F.max("started_at").alias("ultima_viagem"),
    F.sum(F.col("started_at").isNull().cast("int"))
        .alias("nulos_inicio"),
    F.sum(F.col("ended_at").isNull().cast("int"))
        .alias("nulos_fim")
).show(truncate=False)


+---------------+-------------------+-----------------------+------------+---------+
|total_registros|primeira_viagem    |ultima_viagem          |nulos_inicio|nulos_fim|
+---------------+-------------------+-----------------------+------------+---------+
|5860568        |2024-01-01 00:00:39|2024-12-31 23:56:49.854|0           |0        |
+---------------+-------------------+-----------------------+------------+---------+



## 3. Análise exploratória

A exploração é feita com funções do Spark, sem carregar o conjunto completo em Pandas. Valores ausentes e durações anômalas são investigados **antes** da limpeza.

In [6]:
from pyspark.sql import functions as F

# Total de registros
total = df.count()

# Conta os valores nulos por coluna
df_nulos = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

# Exibe quantidade e percentual de nulos
nulos = df_nulos.first().asDict()

for coluna, quantidade in nulos.items():
    percentual = quantidade / total * 100
    print(f"{coluna}: {quantidade:,} ({percentual:.2f}%)")


ride_id: 0 (0.00%)
rideable_type: 0 (0.00%)
started_at: 0 (0.00%)
ended_at: 0 (0.00%)
start_station_name: 1,073,951 (18.33%)
start_station_id: 1,073,951 (18.33%)
end_station_name: 1,104,653 (18.85%)
end_station_id: 1,104,653 (18.85%)
start_lat: 0 (0.00%)
start_lng: 0 (0.00%)
end_lat: 7,232 (0.12%)
end_lng: 7,232 (0.12%)
member_casual: 0 (0.00%)


In [7]:
# Quantidade de viagens por tipo de bicicleta
df.groupBy("rideable_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)


+----------------+-------+
|rideable_type   |count  |
+----------------+-------+
|electric_bike   |2980595|
|classic_bike    |2735636|
|electric_scooter|144337 |
+----------------+-------+



In [8]:
# Quantidade de viagens por perfil de usuário
df.groupBy("member_casual") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()


+-------------+-------+
|member_casual|  count|
+-------------+-------+
|       member|3708910|
|       casual|2151658|
+-------------+-------+



### 3.1 Variável-alvo: duração em minutos

A duração é a diferença entre término e início, em milissegundos, dividida por 60.000. `ended_at` é usado **apenas para construir a variável-alvo**, nunca como preditor.

In [9]:
# Cria a duração em minutos
df = df.withColumn(
    "duracao_minutos",
    (
        F.unix_millis("ended_at") -
        F.unix_millis("started_at")
    ) / 60000.0
)

# Visualiza os primeiros registros
df.select(
    "started_at",
    "ended_at",
    "duracao_minutos"
).show(5, truncate=False)


+-----------------------+-----------------------+-------------------+
|started_at             |ended_at               |duracao_minutos    |
+-----------------------+-----------------------+-------------------+
|2024-06-11 17:20:06.289|2024-06-11 17:21:39.464|1.5529166666666667 |
|2024-06-11 17:19:21.567|2024-06-11 17:19:36.377|0.24683333333333332|
|2024-06-11 17:25:27.089|2024-06-11 17:30:13.035|4.765766666666667  |
|2024-06-11 11:53:50.769|2024-06-11 12:08:13.382|14.376883333333334 |
|2024-06-11 00:11:08.237|2024-06-11 00:11:22.998|0.24601666666666666|
+-----------------------+-----------------------+-------------------+
only showing top 5 rows


In [10]:
# Estatísticas descritivas da duração
df.select("duracao_minutos").describe().show()


+-------+-------------------+
|summary|    duracao_minutos|
+-------+-------------------+
|  count|            5860568|
|   mean| 17.315584450665224|
| stddev|  61.60260659515127|
|    min|-2748.3166666666666|
|    max| 1559.9333333333334|
+-------+-------------------+



In [11]:
# Percentis aproximados da duração
df.select(
    F.percentile_approx(
        "duracao_minutos",
        [0.25, 0.5, 0.75, 0.95, 0.99],
        10000
    ).alias("percentis")
).show(truncate=False)


+-----------------------------------------------------------------+
|percentis                                                        |
+-----------------------------------------------------------------+
|[5.54475, 9.716666666666667, 17.25, 42.734566666666666, 100.8493]|
+-----------------------------------------------------------------+



In [12]:
# Investiga durações potencialmente inválidas
df.agg(
    F.sum(
        (F.col("duracao_minutos") <= 0).cast("int")
    ).alias("duracao_zero_ou_negativa"),

    F.sum(
        (F.col("duracao_minutos") > 1440).cast("int")
    ).alias("duracao_maior_24h"),

    F.sum(
        F.col("duracao_minutos").isNull().cast("int")
    ).alias("duracao_nula")
).show()


+------------------------+-----------------+------------+
|duracao_zero_ou_negativa|duracao_maior_24h|duracao_nula|
+------------------------+-----------------+------------+
|                     723|             7596|           0|
+------------------------+-----------------+------------+



## 4. Limpeza e tratamento dos dados

### 4.1 Valores ausentes

A estação de origem apresenta 18,33% de nomes ausentes; esse campo é preenchido com `Desconhecida`, sem excluir viagens. Os atributos utilizados no modelo não apresentam nulos na exploração inicial. Informações de destino não são utilizadas como preditores e seus nulos são preservados.

In [13]:
# Cria um DataFrame para a etapa de limpeza
df_limpo = df.fillna({
    "start_station_name": "Desconhecida"
})

# Verifica os valores nulos após o tratamento
df_limpo.select(
    F.sum(
        F.col("start_station_name").isNull().cast("int")
    ).alias("nulos_estacao_origem")
).show()


+--------------------+
|nulos_estacao_origem|
+--------------------+
|                   0|
+--------------------+



### 4.2 Inconsistências e deduplicação

Foram identificadas **723 viagens com duração zero/negativa** e **7.596 com duração superior a 24 horas**. Para a modelagem, restringimos a duração ao intervalo `(0, 1440]` minutos. O teto de 24 horas é um **critério operacional**, e não a comprovação de que viagens mais longas sejam falsas.

Encontramos identificadores `ride_id` repetidos sem linhas inteiramente iguais. Um exemplo mostrou a mesma viagem registrada com e sem milissegundos, possivelmente na sobreposição entre arquivos mensais. A deduplicação mantém **um registro por ID**, priorizando horários com maior precisão; isso não garante qual versão é factualmente correta e constitui uma limitação da limpeza.

In [14]:
# Conta registros e identificadores distintos
df_limpo.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("ride_id").alias("ids_distintos")
).show()


+---------------+-------------+
|total_registros|ids_distintos|
+---------------+-------------+
|        5860568|      5860357|
+---------------+-------------+



In [15]:
from pyspark.sql.window import Window

# Filtra apenas durações admissíveis para o projeto.
df_validos = df.filter(
    (F.col("duracao_minutos") > 0) &
    (F.col("duracao_minutos") <= 1440)
)

# Prioriza registros que preservam frações de segundo.
df_validos = df_validos.withColumn(
    "precisao",
    F.when(
        (F.unix_millis("started_at") % 1000 != 0) |
        (F.unix_millis("ended_at") % 1000 != 0), 1
    ).otherwise(0)
)

janela = Window.partitionBy("ride_id").orderBy(
    F.desc("precisao"), F.desc("started_at"), F.desc("ended_at")
)

df_limpo = (
    df_validos
    .withColumn("ordem", F.row_number().over(janela))
    .filter(F.col("ordem") == 1)
    .drop("ordem", "precisao")
    .fillna({"start_station_name": "Desconhecida"})
)


In [16]:
# Quantidade final
print("Registros após limpeza:", df_limpo.count())

# IDs duplicados restantes
duplicados = (
    df_limpo.count()
    - df_limpo.select("ride_id").distinct().count()
)

print("IDs duplicados restantes:", duplicados)


Registros após limpeza: 5852081
IDs duplicados restantes: 0


A limpeza preservou **5.852.081 viagens**, com **zero identificadores repetidos**, removendo **8.487 registros** em relação à base original.

### 4.3 Armazenamento Parquet

Após a limpeza, os dados são salvos em Parquet, particionados por mês do início da viagem. Esse formato mantém os tipos e facilita leituras seletivas por período.

In [17]:
df_limpo = df_limpo.withColumn(
    "mes",
    F.month("started_at")
)

# Quantidade de viagens por mês
df_limpo.groupBy("mes") \
    .count() \
    .orderBy("mes") \
    .show(12)


+---+------+
|mes| count|
+---+------+
|  1|144447|
|  2|222768|
|  3|301166|
|  4|414320|
|  5|608407|
|  6|709553|
|  7|747838|
|  8|754801|
|  9|820136|
| 10|615688|
| 11|334734|
| 12|178223|
+---+------+



In [18]:
caminho_parquet = "/content/divvy_parquet"

# Grava os dados particionados por mês
(
    df_limpo.write
    .mode("overwrite")
    .partitionBy("mes")
    .parquet(caminho_parquet)
)

print("Dados salvos com sucesso!")


Dados salvos com sucesso!


In [19]:
# Recupera os dados armazenados
df_parquet = spark.read.parquet(caminho_parquet)

# Confere a quantidade de registros
print("Registros originais:", df_limpo.count())
print("Registros em Parquet:", df_parquet.count())


Registros originais: 5852081
Registros em Parquet: 5852081


## 5. Engenharia de atributos e divisão dos dados

Extraímos `hora`, `dia_semana`, `fim_semana` e utilizamos `mes`, coordenadas de origem, tipo de veículo e perfil de usuário. Não utilizamos informações de destino, que não estão disponíveis no início da viagem.

In [20]:
df_modelo = (
    df_parquet
    .withColumn("hora", F.hour("started_at"))
    .withColumn("dia_semana", F.dayofweek("started_at"))
    .withColumn(
        "fim_semana",
        F.when(F.col("dia_semana").isin(1, 7), 1).otherwise(0)
    )
    .select(
        "started_at", "rideable_type", "member_casual",
        "hora", "dia_semana", "fim_semana", "mes",
        "start_lat", "start_lng", "duracao_minutos"
    )
)

df_modelo.printSchema()


root
 |-- started_at: timestamp (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- hora: integer (nullable = true)
 |-- dia_semana: integer (nullable = true)
 |-- fim_semana: integer (nullable = false)
 |-- mes: integer (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- duracao_minutos: double (nullable = true)



### 5.1 Divisão cronológica

**Treino:** janeiro–setembro de 2024. **Teste:** outubro–dezembro de 2024. A divisão não é exatamente 80/20 porque o volume varia entre meses. O conjunto de teste não é usado para ajustar o Pipeline.

In [21]:
# Treino: janeiro a setembro
df_treino = df_modelo.filter(
    F.col("mes") <= 9
)

# Teste: outubro a dezembro
df_teste = df_modelo.filter(
    F.col("mes") >= 10
)

# Verifica as quantidades
print("Treino:", df_treino.count())
print("Teste:", df_teste.count())


Treino: 4723436
Teste: 1128645


In [22]:
# Verifica o intervalo temporal dos conjuntos
for nome, dados in [
    ("Treino", df_treino),
    ("Teste", df_teste)
]:
    print(nome)
    dados.agg(
        F.min("started_at").alias("inicio"),
        F.max("started_at").alias("fim")
    ).show(truncate=False)


Treino
+-------------------+-----------------------+
|inicio             |fim                    |
+-------------------+-----------------------+
|2024-01-01 00:00:39|2024-09-30 23:59:45.253|
+-------------------+-----------------------+

Teste
+-----------------------+-----------------------+
|inicio                 |fim                    |
+-----------------------+-----------------------+
|2024-10-01 00:00:38.341|2024-12-31 23:56:49.854|
+-----------------------+-----------------------+



### 5.2 Pipeline de pré-processamento

`StringIndexer` e `OneHotEncoder` transformam categorias; `VectorAssembler` reúne atributos; `StandardScaler` padroniza o vetor de entrada. O ajuste acontece apenas durante o `fit(df_treino)`.

In [23]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Converte categorias em índices numéricos
indexador = StringIndexer(
    inputCols=["rideable_type", "member_casual"],
    outputCols=["rideable_idx", "usuario_idx"],
    handleInvalid="keep"
)

# Codifica os índices em vetores
codificador = OneHotEncoder(
    inputCols=["rideable_idx", "usuario_idx"],
    outputCols=["rideable_ohe", "usuario_ohe"]
)


In [24]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler
)

# Variáveis numéricas
colunas_numericas = [
    "hora",
    "dia_semana",
    "fim_semana",
    "mes",
    "start_lat",
    "start_lng"
]

# Reúne variáveis numéricas e categóricas codificadas
assembler = VectorAssembler(
    inputCols=colunas_numericas + [
        "rideable_ohe",
        "usuario_ohe"
    ],
    outputCol="features_brutas"
)

# Padroniza os atributos
scaler = StandardScaler(
    inputCol="features_brutas",
    outputCol="features",
    withMean=False,
    withStd=True
)

# Organiza as etapas do Pipeline
pipeline_preprocessamento = Pipeline(
    stages=[
        indexador,
        codificador,
        assembler,
        scaler
    ]
)

print("Pipeline criado com sucesso!")


Pipeline criado com sucesso!


## 6. Modelagem preditiva

### 6.1 Baseline

Uma previsão constante pela **média do treinamento** fornece a referência para avaliar os modelos de regressão, sem utilizar os rótulos do conjunto de teste no cálculo dessa média.

In [25]:
# Calcula a média apenas com os dados de treinamento
media_treino = df_treino.agg(
    F.avg("duracao_minutos")
).first()[0]

print(f"Duração média do treino: {media_treino:.2f} minutos")


Duração média do treino: 15.90 minutos


In [26]:
# Cria previsões constantes com a média do treino
df_baseline = df_teste.withColumn(
    "prediction",
    F.lit(media_treino)
)

# Compara valores reais e previstos
df_baseline.select(
    "duracao_minutos",
    "prediction"
).show(10)


+------------------+------------------+
|   duracao_minutos|        prediction|
+------------------+------------------+
| 5.323033333333333|15.896285545878687|
|18.393483333333332|15.896285545878687|
|           17.5336|15.896285545878687|
| 9.232566666666667|15.896285545878687|
|          15.35555|15.896285545878687|
|            2.2999|15.896285545878687|
|53.081916666666665|15.896285545878687|
| 6.181633333333333|15.896285545878687|
|            6.2943|15.896285545878687|
|          16.09195|15.896285545878687|
+------------------+------------------+
only showing top 10 rows


In [27]:
from pyspark.ml.evaluation import RegressionEvaluator

# Avalia o baseline com três métricas
metricas = {}

for metrica in ["rmse", "mae", "r2"]:

    avaliador = RegressionEvaluator(
        labelCol="duracao_minutos",
        predictionCol="prediction",
        metricName=metrica
    )

    metricas[metrica] = avaliador.evaluate(df_baseline)

# Exibe os resultados
for nome, valor in metricas.items():
    print(f"{nome.upper()}: {valor:.4f}")


RMSE: 28.4107
MAE: 10.7004
R2: -0.0085


### 6.2 Regressão Linear

O algoritmo `LinearRegression` é integrado ao Pipeline completo. Foram usados `maxIter=50` e `regParam=0.1`, como configuração inicial priorizando simplicidade. **Não houve busca sistemática de hiperparâmetros**, o que será considerado uma limitação.

In [28]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

# Define o algoritmo de regressão
regressao_linear = LinearRegression(
    featuresCol="features",
    labelCol="duracao_minutos",
    predictionCol="prediction",
    maxIter=50,
    regParam=0.1
)

# Integra pré-processamento e regressão
pipeline_lr = Pipeline(
    stages=pipeline_preprocessamento.getStages() +
           [regressao_linear]
)


In [29]:
# Treina o Pipeline apenas com dados de treinamento
modelo_lr = pipeline_lr.fit(df_treino)

print("Regressão Linear treinada com sucesso!")


Regressão Linear treinada com sucesso!


In [30]:
# Aplica o modelo aos dados de teste
previsoes_lr = modelo_lr.transform(df_teste)

# Compara valores reais e previstos
previsoes_lr.select(
    "duracao_minutos",
    "prediction"
).show(10)


+------------------+------------------+
|   duracao_minutos|        prediction|
+------------------+------------------+
| 5.323033333333333| 8.695756097737103|
|18.393483333333332|12.658722033950198|
|           17.5336| 16.66688489975286|
| 9.232566666666667| 9.616025676247773|
|          15.35555|10.461140894910386|
|            2.2999|17.055548171534156|
|53.081916666666665|19.657693084676794|
| 6.181633333333333|10.521266552374527|
|            6.2943|28.156009395791898|
|          16.09195|26.087791997531895|
+------------------+------------------+
only showing top 10 rows


In [31]:
# Calcula RMSE, MAE e R²
metricas_lr = {}

for metrica in ["rmse", "mae", "r2"]:

    avaliador = RegressionEvaluator(
        labelCol="duracao_minutos",
        predictionCol="prediction",
        metricName=metrica
    )

    metricas_lr[metrica] = avaliador.evaluate(previsoes_lr)

for nome, valor in metricas_lr.items():
    print(f"{nome.upper()}: {valor:.4f}")


RMSE: 28.1528
MAE: 10.7441
R2: 0.0097


### 6.3 Random Forest Regressor

O `RandomForestRegressor` também é um **algoritmo de regressão**: combina árvores de decisão para prever valores numéricos. Para comparar os algoritmos, utilizamos as mesmas variáveis e a mesma divisão cronológica.

Os transformadores já ajustados **somente no treino** pelo Pipeline da Regressão Linear serão reutilizados. Assim, não precisamos reajustar `StringIndexer`, `OneHotEncoder` e `StandardScaler`. Configuração inicial: 20 árvores, profundidade máxima de 5 e amostragem de 50% por árvore, para reduzir o custo de treinamento no Colab. Não houve busca sistemática de hiperparâmetros.

In [32]:
from pyspark.ml.regression import RandomForestRegressor

# As etapas anteriores do modelo linear são transformadores já ajustados no treino.
# Não usamos informações do teste para ajustar o pré-processamento.
preprocessamento_ajustado = modelo_lr.stages[:-1]

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="duracao_minutos",
    predictionCol="prediction",
    numTrees=20,
    maxDepth=5,
    subsamplingRate=0.5,
    seed=42
)

# Pipeline completo para a Random Forest, reutilizando o pré-processamento.
pipeline_rf = Pipeline(stages=preprocessamento_ajustado + [rf])
modelo_rf = pipeline_rf.fit(df_treino)

print("Random Forest treinada com sucesso!")

Random Forest treinada com sucesso!


In [33]:
# Previsões e métricas no mesmo conjunto de teste usado pela Regressão Linear.
previsoes_rf = modelo_rf.transform(df_teste)

metricas_rf = {}
for metrica in ["rmse", "mae", "r2"]:
    avaliador = RegressionEvaluator(
        labelCol="duracao_minutos",
        predictionCol="prediction",
        metricName=metrica
    )
    metricas_rf[metrica] = avaliador.evaluate(previsoes_rf)

for nome, valor in metricas_rf.items():
    print(f"Random Forest — {nome.upper()}: {valor:.4f}")

previsoes_rf.select("duracao_minutos", "prediction").show(5)

Random Forest — RMSE: 27.8114
Random Forest — MAE: 9.8357
Random Forest — R2: 0.0336
+------------------+------------------+
|   duracao_minutos|        prediction|
+------------------+------------------+
| 5.323033333333333|11.294513909260377|
|18.393483333333332|11.726786270907645|
|           17.5336|13.284347192279029|
| 9.232566666666667|10.864955399531603|
|          15.35555|12.931200000264019|
+------------------+------------------+
only showing top 5 rows


## 7. Avaliação e persistência dos modelos

### 7.1 Comparação no conjunto de teste

Comparamos **baseline**, **Regressão Linear** e **Random Forest Regressor** no mesmo período (outubro–dezembro de 2024). RMSE e MAE são expressos em minutos e devem ser **menores**; R² deve ser **maior**. A comparação é descritiva: o conjunto de teste não foi utilizado para ajustar hiperparâmetros.

O baseline sempre prevê a duração média observada no conjunto de treinamento. Como os modelos não recebem o destino planejado nem informações futuras da viagem, seu desempenho pode ser limitado. A divisão cronológica também deixa o teste sujeito a diferenças sazonais. Os parâmetros de ambos os algoritmos foram definidos inicialmente, sem busca sistemática de hiperparâmetros.

In [34]:
# Compara os resultados medidos, sem valores pré-preenchidos para a Random Forest.
resultados = [
    ("Baseline", metricas),
    ("Regressão Linear", metricas_lr),
    ("Random Forest", metricas_rf),
]

print(f"{'Modelo':<20} {'RMSE (min)':>12} {'MAE (min)':>12} {'R²':>10}")
print("-" * 58)
for nome, m in resultados:
    print(f"{nome:<20} {m['rmse']:>12.4f} {m['mae']:>12.4f} {m['r2']:>10.4f}")

# Diferença de desempenho da Random Forest em relação ao baseline.
print("\nVariação do RMSE da Random Forest vs baseline:",
      f"{(metricas_rf['rmse'] / metricas['rmse'] - 1) * 100:+.2f}%")

Modelo                 RMSE (min)    MAE (min)         R²
----------------------------------------------------------
Baseline                  28.4107      10.7004    -0.0085
Regressão Linear          28.1528      10.7441     0.0097
Random Forest             27.8114       9.8357     0.0336

Variação do RMSE da Random Forest vs baseline: -2.11%


### 7.2 Persistência dos modelos

Os dois Pipelines incluem o pré-processamento ajustado no treinamento e o respectivo regressor. São salvos com `save()` e podem ser carregados posteriormente com `PipelineModel.load()`. As pastas locais são regeneradas ao reexecutar o notebook e **não precisam ser enviadas ao GitHub**.

In [35]:
# Define o caminho para salvar o modelo
caminho_modelo = "/content/modelo_regressao_divvy"

# Salva o Pipeline treinado
modelo_lr.write().overwrite().save(caminho_modelo)

print("Modelo salvo com sucesso!")


Modelo salvo com sucesso!


In [36]:
from pyspark.ml import PipelineModel

# Carrega o modelo salvo
modelo_carregado = PipelineModel.load(caminho_modelo)

# Testa o modelo com 5 registros
amostra = df_teste.limit(5)

previsoes = modelo_carregado.transform(amostra)

previsoes.select(
    "duracao_minutos",
    "prediction"
).show()


+------------------+------------------+
|   duracao_minutos|        prediction|
+------------------+------------------+
| 5.323033333333333| 8.695756097737103|
|18.393483333333332|12.658722033950198|
|           17.5336| 16.66688489975286|
| 9.232566666666667| 9.616025676247773|
|          15.35555|10.461140894910386|
+------------------+------------------+



In [37]:
# Salva e recarrega o Pipeline da Random Forest.
caminho_modelo_rf = "/content/modelo_random_forest_divvy"
modelo_rf.write().overwrite().save(caminho_modelo_rf)
modelo_rf_carregado = PipelineModel.load(caminho_modelo_rf)

# Confirma que o modelo recuperado consegue prever.
modelo_rf_carregado.transform(df_teste.limit(5)).select(
    "duracao_minutos", "prediction"
).show()
print("Random Forest salva e carregada com sucesso!")

+------------------+------------------+
|   duracao_minutos|        prediction|
+------------------+------------------+
| 5.323033333333333|11.294513909260377|
|18.393483333333332|11.726786270907645|
|           17.5336|13.284347192279029|
| 9.232566666666667|10.864955399531603|
|          15.35555|12.931200000264019|
+------------------+------------------+

Random Forest salva e carregada com sucesso!


## 8. Conclusões

Este projeto utilizou **5.860.568 registros** de viagens da Divvy em 2024. Após a limpeza, foram mantidos **5.852.081 registros**. Com o PySpark, realizamos leitura dos arquivos, tratamento dos dados, criação de atributos e armazenamento em Parquet. Também construímos um Pipeline do Spark ML para treinar modelos de regressão.

O objetivo foi prever a **duração da viagem, em minutos**, usando informações disponíveis no início do percurso, como horário, tipo de veículo, perfil de usuário e localização de origem. Para avaliar os modelos, utilizamos viagens de janeiro a setembro no treinamento e de outubro a dezembro no teste.

Comparamos uma previsão simples de referência (**baseline**, que sempre prevê a média de **15,90 minutos** do treinamento), a **Regressão Linear** e a **Random Forest**. Os resultados obtidos no conjunto de teste foram:

| Modelo | RMSE (min) | MAE (min) | R² |
|---|---:|---:|---:|
| Baseline | 28,4107 | 10,7004 | -0,0085 |
| Regressão Linear | 28,1528 | 10,7441 | 0,0097 |
| **Random Forest** | **27,8114** | **9,8357** | **0,0336** |

A **Random Forest apresentou o melhor resultado** entre os modelos testados. Em relação ao baseline, reduziu o erro absoluto médio (MAE) de **10,70 para 9,84 minutos**, uma melhoria de aproximadamente **8,08%**. Mesmo assim, seu **R² de 0,0336** mostra que ela explicou apenas cerca de **3,36% da variação** na duração das viagens. Portanto, a melhoria foi observada nas métricas do teste, mas a **capacidade preditiva ainda é baixa**.

Uma possível explicação é que o modelo não conhece o destino nem a distância planejada da viagem, fatores que provavelmente influenciam bastante a duração. Além disso, horários semelhantes podem corresponder a trajetos muito diferentes. Como o objetivo foi prever no início da viagem sem conhecer o destino, essas informações não foram utilizadas. Também não houve busca sistemática de hiperparâmetros, e a avaliação foi feita em apenas um período de teste.

Assim, **o projeto cumpriu o objetivo acadêmico de construir, avaliar e salvar modelos de regressão com Spark ML**, mas os resultados não são suficientes para recomendar o uso operacional das previsões. Como melhorias futuras, seria possível testar outras configurações dos modelos, avaliar novos períodos e, caso o usuário informe o destino antes de partir, incluir uma estimativa de distância entre origem e destino.
